# AE2: manejo de archivos CSV, JSON y TXT

Continuación de la actividad 1: en lugar de cargar el dataset con la librería `datasets`, ahora se descargan manualmente un CSV y un JSON de Hugging Face y se manejan con tres clases.

```
Descarga manual → importar → modificar → guardar nuevo CSV y JSON
                → GeneradorResumenTXT → resumen.txt con URL de origen
```

| Archivo | Clase | Qué hace |
|---|---|---|
| `manejador_csv.py` | `ManejadorCSV` | importa con `csv.reader`, actualiza un valor y guarda con `csv.writer` |
| `manejador_json.py` | `ManejadorJSON` | importa con `json.load`, actualiza un valor y guarda con `json.dump` |
| `generador_resumen_txt.py` | `GeneradorResumenTXT` | genera `resumen.txt` con los resultados de las otras dos clases |

## 1. URL de origen de los archivos

Los archivos se descargaron manualmente desde Hugging Face y se guardaron en la carpeta `datos/`:

- CSV: [RummageLabs/pixar_movies → pixar_movies.csv](https://huggingface.co/datasets/RummageLabs/pixar_movies/blob/main/pixar_movies.csv) → `datos/pixar_movies.csv` (28 películas de Pixar)
- JSON: [llamafactory/tiny-supervised-dataset → train.json](https://huggingface.co/datasets/llamafactory/tiny-supervised-dataset/blob/main/train.json) → `datos/train.json` (300 instrucciones)

Las mismas URL se guardan en la variable `urls` para enviarlas al resumen.

In [1]:
urls = {
    "CSV": "https://huggingface.co/datasets/RummageLabs/pixar_movies/blob/main/pixar_movies.csv",
    "JSON": "https://huggingface.co/datasets/llamafactory/tiny-supervised-dataset/blob/main/train.json",
}
urls

{'CSV': 'https://huggingface.co/datasets/RummageLabs/pixar_movies/blob/main/pixar_movies.csv',
 'JSON': 'https://huggingface.co/datasets/llamafactory/tiny-supervised-dataset/blob/main/train.json'}

## 2. Importar las clases CSV y JSON e inicializar sus objetos

In [2]:
from manejador_csv import ManejadorCSV
from manejador_json import ManejadorJSON

manejador_csv = ManejadorCSV("datos/pixar_movies.csv")
manejador_json = ManejadorJSON("datos/train.json")

## 3. Importar los archivos locales

In [3]:
filas = manejador_csv.importar()

print("Encabezados:", manejador_csv.encabezados)
print("Registros CSV:", len(filas))
filas[0]

Encabezados: ['movie', 'date_released', 'year_released', 'length_min', 'plot_summary', 'director', 'writer', 'main_characters', 'type_of_characters', 'main_voice_actors', 'opening_weekend_box_office_sales', 'total_worldwide_gross_sales', 'rotten_tomatoes_rating', 'imdb_rating', 'movie_genre', 'movie_rating']
Registros CSV: 28


['Toy Story',
 '1995-11-22',
 '1995',
 '81',
 'A group of toys, headed by Woody, a pull-string cowboy doll, fear being replaced when a new action figure, Buzz Lightyear, comes into the picture. When they accidentally get lost, Woody and Buzz must work together to return to their owner, Andy, while overcoming their rivalry and learning the value of friendship.',
 'John Lasseter',
 'John Lasseter, Pete Docter, Andrew Stanton, Joe Ranft, Joss Whdeon, Joel Cohen, Alec Sokolow',
 'Woody, Buzz Lightyear, Andy, Mr. Potato Head, Rex, Hamm',
 'Toys, Humans',
 'Tom Hanks, Tim Allen, Don Rickles, Jim Varney',
 '$29.1 million',
 '$394.4 million',
 '100%',
 '8.3/10',
 'Animation, Adventure, Comedy, Family',
 'G']

In [4]:
registros = manejador_json.importar()

print("Registros JSON:", len(registros))
registros[:2]

Registros JSON: 300


[{'instruction': 'Give three tips for staying healthy.',
  'input': '',
  'output': '1. Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'},
 {'instruction': 'What are the three primary colors?',
  'input': '',
  'output': 'The three primary colors are red, blue, and yellow.'}]

## 4. Modificar los datos con los métodos de cada clase

### CSV: actualizar un valor

Se traduce al español el título de la película de la fila 4 (*Finding Nemo*).

In [5]:
print("Antes:  ", manejador_csv.filas[4][0])

manejador_csv.actualizar_valor(4, "movie", "Buscando a Nemo")

print("Después:", manejador_csv.filas[4][0])

Antes:   Finding Nemo
Después: Buscando a Nemo


### JSON: actualizar un valor

Se traduce al español la instrucción del primer registro.

In [6]:
print("Antes:  ", manejador_json.registros[0]["instruction"])

manejador_json.actualizar_valor(0, "instruction", "Da tres consejos para mantenerse saludable.")

print("Después:", manejador_json.registros[0]["instruction"])

Antes:   Give three tips for staying healthy.
Después: Da tres consejos para mantenerse saludable.


## 5. Guardar el nuevo CSV y el nuevo JSON

Se guardan con nombres nuevos para conservar los archivos originales.

In [7]:
ruta_csv = manejador_csv.guardar("datos/pixar_movies_modificado.csv")
ruta_json = manejador_json.guardar("datos/train_modificado.json")

print("Nuevo CSV: ", ruta_csv)
print("Nuevo JSON:", ruta_json)

Nuevo CSV:  datos/pixar_movies_modificado.csv
Nuevo JSON: datos/train_modificado.json


Verificación: se importan los archivos nuevos con las mismas clases y se comparan con los datos modificados.

In [8]:
nuevo_csv = ManejadorCSV(ruta_csv)
nuevo_csv.importar()
print("CSV guardado:", len(nuevo_csv.filas), "registros. Película 4:", nuevo_csv.filas[4][0])
print("¿Contiene los cambios?", nuevo_csv.filas == manejador_csv.filas)

nuevo_json = ManejadorJSON(ruta_json)
nuevo_json.importar()
print("JSON guardado:", len(nuevo_json.registros), "registros. Instrucción 0:", nuevo_json.registros[0]["instruction"])
print("¿Contiene los cambios?", nuevo_json.registros == manejador_json.registros)

CSV guardado: 28 registros. Película 4: Buscando a Nemo
¿Contiene los cambios? True
JSON guardado: 300 registros. Instrucción 0: Da tres consejos para mantenerse saludable.
¿Contiene los cambios? True


## 6. Generar el resumen .txt con la tercera clase

In [9]:
from generador_resumen_txt import GeneradorResumenTXT

generador = GeneradorResumenTXT(urls, manejador_csv.resultados(), manejador_json.resultados())
ruta_resumen = generador.generar("resumen.txt")
print("Archivo generado:", ruta_resumen)

Archivo generado: resumen.txt


Contenido del archivo `resumen.txt`:

In [10]:
with open(ruta_resumen, encoding="utf-8") as archivo:
    print(archivo.read())

Resumen AE2: manejo de archivos CSV, JSON y TXT

Archivo CSV
  URL de origen: https://huggingface.co/datasets/RummageLabs/pixar_movies/blob/main/pixar_movies.csv
  Archivo importado: datos/pixar_movies.csv
  Registros importados: 28
  Modificaciones realizadas:
    - Fila 4: la columna 'movie' cambió de 'Finding Nemo' a 'Buscando a Nemo'
  Nuevo archivo guardado: datos/pixar_movies_modificado.csv
  Registros guardados: 28

Archivo JSON
  URL de origen: https://huggingface.co/datasets/llamafactory/tiny-supervised-dataset/blob/main/train.json
  Archivo importado: datos/train.json
  Registros importados: 300
  Modificaciones realizadas:
    - Registro 0: el campo 'instruction' cambió de 'Give three tips for staying healthy.' a 'Da tres consejos para mantenerse saludable.'
  Nuevo archivo guardado: datos/train_modificado.json
  Registros guardados: 300

Clases y métodos utilizados
  - ManejadorCSV: importar, actualizar_valor, guardar, resultados
  - ManejadorJSON: importar, actualizar_valo